# 🌡️ Temperature Sensor Fault Detection
## Phase 4: Model Training & Evaluation

---
**Goal:** Train machine learning models to classify the 4 fault classes, evaluate their performance, and pick the best one  
**Input:** `temp_features.parquet` + `temp_splits.pkl` from Phase 3  
**Output:** Best trained model + full evaluation report

### Models We Train (in order of complexity)
| # | Model | Type | Notes |
|---|---|---|---|
| 1 | Random Forest | Classical (tree ensemble) | Fast, interpretable, no scaling needed |
| 2 | K-Nearest Neighbors | Classical (distance-based) | Simple, needs scaled features |
| 3 | Support Vector Machine | Classical (margin-based) | Great for small feature sets, needs scaling |
| 4 | XGBoost | Gradient boosting | Usually highest accuracy |

### Sections
1. Imports & Load Phase 3 Data  
2. Model 1 — Random Forest  
3. Model 2 — K-Nearest Neighbors  
4. Model 3 — Support Vector Machine  
5. Model 4 — XGBoost  
6. Model Comparison  
7. Best Model — Deep Dive Evaluation  
8. Feature Importance  
9. Save Best Model  

---
## 1. Imports & Load Phase 3 Data

Install XGBoost if needed:
```
pip install xgboost scikit-learn
```

In [ ]:
import os, json, pickle, warnings, time
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.ensemble        import RandomForestClassifier
from sklearn.neighbors       import KNeighborsClassifier
from sklearn.svm             import SVC
from sklearn.metrics         import (classification_report, confusion_matrix,
                                      accuracy_score, f1_score,
                                      roc_auc_score, roc_curve,
                                      ConfusionMatrixDisplay)
from sklearn.model_selection import cross_val_score, StratifiedKFold

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
    print('✅ XGBoost available')
except ImportError:
    XGBOOST_AVAILABLE = False
    print('⚠️  XGBoost not found — run: pip install xgboost')

warnings.filterwarnings('ignore')
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

CLASS_COLORS = {
    'normal'        : '#43A047',
    'noise_fault'   : '#E53935',
    'bias_fault'    : '#FB8C00',
    'flatline_fault': '#8E24AA',
}
MODEL_COLORS = ['#1565C0','#2E7D32','#E65100','#6A1B9A']

print('✅ Imports done.')

In [ ]:
# ================================================================
#  ONLY CHANGE THIS if your path is different
# ================================================================
BASE_PATH  = r'C:\PVS_Dataset'
PHASE3_DIR = os.path.join(BASE_PATH, 'outputs_temp_phase3')
OUTPUT_DIR = os.path.join(BASE_PATH, 'outputs_temp_phase4')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load Phase 3 config
with open(os.path.join(PHASE3_DIR, 'temp_phase3_config.json')) as f:
    cfg = json.load(f)

FINAL_FEATS  = cfg['FINAL_FEATS']
CLASS_LABELS = {int(k): v for k, v in cfg['CLASS_LABELS'].items()}
CLASS_NAMES  = [CLASS_LABELS[i] for i in sorted(CLASS_LABELS.keys())]

# Load train/test splits
with open(cfg['splits_path'], 'rb') as f:
    splits = pickle.load(f)

X_train   = splits['X_train']
X_test    = splits['X_test']
X_train_s = splits['X_train_s']   # scaled — for SVM, KNN
X_test_s  = splits['X_test_s']
y_train   = splits['y_train']
y_test    = splits['y_test']

print('✅ Phase 3 data loaded:')
print(f'   Features     : {FINAL_FEATS}')
print(f'   Classes      : {CLASS_NAMES}')
print(f'   X_train      : {X_train.shape}')
print(f'   X_test       : {X_test.shape}')

In [ ]:
# ── Helper: evaluate any trained model ──
results_log = []   # stores summary for all models

def evaluate_model(model, X_tr, X_te, y_tr, y_te, model_name):
    """
    Train, predict, and compute all metrics for one model.
    Returns a result dict and prints a summary.
    """
    t0 = time.time()
    model.fit(X_tr, y_tr)
    train_time = time.time() - t0

    t1 = time.time()
    y_pred = model.predict(X_te)
    infer_time = (time.time() - t1) / len(X_te) * 1000  # ms per sample

    acc    = accuracy_score(y_te, y_pred)
    f1_mac = f1_score(y_te, y_pred, average='macro')
    f1_wtd = f1_score(y_te, y_pred, average='weighted')

    # Cross-val on training set
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X_tr, y_tr, cv=cv, scoring='f1_macro', n_jobs=-1)

    result = {
        'Model'         : model_name,
        'Test Accuracy' : round(acc * 100, 2),
        'F1 Macro'      : round(f1_mac * 100, 2),
        'F1 Weighted'   : round(f1_wtd * 100, 2),
        'CV F1 Mean'    : round(cv_scores.mean() * 100, 2),
        'CV F1 Std'     : round(cv_scores.std()  * 100, 2),
        'Train Time (s)': round(train_time, 2),
        'Infer (ms/smpl)': round(infer_time, 4),
        'y_pred'        : y_pred,
        'model_obj'     : model,
    }

    print(f'\n{'='*55}')
    print(f'  {model_name}')
    print(f'{'='*55}')
    print(f'  Test Accuracy : {acc*100:.2f}%')
    print(f'  F1 Macro      : {f1_mac*100:.2f}%')
    print(f'  F1 Weighted   : {f1_wtd*100:.2f}%')
    print(f'  CV F1 (5-fold): {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%')
    print(f'  Train time    : {train_time:.2f}s')
    print(f'  Inference     : {infer_time:.4f} ms/sample')
    print(f'\n  Per-class report:')
    print(classification_report(y_te, y_pred,
                                 target_names=CLASS_NAMES, digits=4))
    return result


def plot_confusion(y_true, y_pred, model_name, fname=None):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(8, 6))
    disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
    disp.plot(cmap='Blues', ax=ax, colorbar=False)
    ax.set_title(f'Confusion Matrix — {model_name}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    if fname:
        plt.savefig(os.path.join(OUTPUT_DIR, fname), dpi=150, bbox_inches='tight')
        print(f'📁 Saved: {fname}')
    plt.show()

print('✅ Helper functions ready.')

---
## 2. Model 1 — Random Forest

**Best starting model.** Uses multiple decision trees and votes on the answer.  
- Does NOT need scaled features  
- Gives feature importance scores  
- Robust to outliers

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators = 200,      # 200 trees
    max_depth    = None,     # grow until pure leaves
    min_samples_split = 5,
    min_samples_leaf  = 2,
    class_weight = 'balanced',
    random_state = 42,
    n_jobs       = -1,       # use all CPU cores
)

rf_result = evaluate_model(rf_model, X_train, X_test,
                             y_train, y_test, 'Random Forest')
results_log.append(rf_result)
plot_confusion(y_test, rf_result['y_pred'], 'Random Forest', 'model_01_rf_cm.png')

---
## 3. Model 2 — K-Nearest Neighbors

**Simplest possible model.** Labels a window by looking at its K nearest neighbors in feature space.  
- Needs scaled features  
- No training — just stores all training points  
- Good baseline for comparison

In [ ]:
knn_model = KNeighborsClassifier(
    n_neighbors = 7,
    metric      = 'euclidean',
    weights     = 'distance',   # closer neighbors vote more
    n_jobs      = -1,
)

knn_result = evaluate_model(knn_model, X_train_s, X_test_s,
                              y_train, y_test, 'KNN (k=7)')
results_log.append(knn_result)
plot_confusion(y_test, knn_result['y_pred'], 'KNN (k=7)', 'model_02_knn_cm.png')

---
## 4. Model 3 — Support Vector Machine

**Strong classical model.** Finds the widest margin boundary between classes.  
- Needs scaled features  
- RBF kernel handles non-linear boundaries  
- Can be slow on very large datasets

In [ ]:
# SVM can be slow on large datasets
# We subsample training data if > 100k samples to keep it fast
MAX_SVM_TRAIN = 80_000

if len(X_train_s) > MAX_SVM_TRAIN:
    print(f'⚠️  Training set large ({len(X_train_s):,}). Subsampling to {MAX_SVM_TRAIN:,} for SVM.')
    idx = np.random.choice(len(X_train_s), MAX_SVM_TRAIN, replace=False)
    X_svm, y_svm = X_train_s[idx], y_train[idx]
else:
    X_svm, y_svm = X_train_s, y_train

svm_model = SVC(
    C            = 10,
    kernel       = 'rbf',
    gamma        = 'scale',
    class_weight = 'balanced',
    probability  = True,
    random_state = 42,
)

svm_result = evaluate_model(svm_model, X_svm, X_test_s,
                              y_svm, y_test, 'SVM (RBF kernel)')
results_log.append(svm_result)
plot_confusion(y_test, svm_result['y_pred'], 'SVM (RBF)', 'model_03_svm_cm.png')

---
## 4. Model 4 — XGBoost

**Usually the top performer.** Builds trees sequentially, each one correcting the errors of the previous.  
- Does NOT need scaled features  
- Handles class imbalance well  
- Also gives feature importance

In [ ]:
if XGBOOST_AVAILABLE:
    xgb_model = XGBClassifier(
        n_estimators      = 300,
        max_depth         = 6,
        learning_rate     = 0.1,
        subsample         = 0.8,
        colsample_bytree  = 0.8,
        use_label_encoder = False,
        eval_metric       = 'mlogloss',
        random_state      = 42,
        n_jobs            = -1,
    )
    xgb_result = evaluate_model(xgb_model, X_train, X_test,
                                  y_train, y_test, 'XGBoost')
    results_log.append(xgb_result)
    plot_confusion(y_test, xgb_result['y_pred'], 'XGBoost', 'model_04_xgb_cm.png')
else:
    print('⚠️  XGBoost not installed. Skipping.')
    print('   Run: pip install xgboost   then re-run this cell.')

---
## 6. Model Comparison

Side-by-side comparison of all trained models.

In [ ]:
comp_cols = ['Model','Test Accuracy','F1 Macro','F1 Weighted',
              'CV F1 Mean','CV F1 Std','Train Time (s)','Infer (ms/smpl)']
comp_df   = pd.DataFrame([{k: r[k] for k in comp_cols} for r in results_log])
comp_df   = comp_df.sort_values('F1 Macro', ascending=False).reset_index(drop=True)

print('📊 MODEL COMPARISON TABLE')
print('='*85)
display(comp_df)

best_model_name = comp_df.iloc[0]['Model']
best_f1         = comp_df.iloc[0]['F1 Macro']
print(f'\n🏆 Best model: {best_model_name}  (F1 Macro = {best_f1:.2f}%)')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 6))

models    = comp_df['Model'].tolist()
n_models  = len(models)
bar_colors = MODEL_COLORS[:n_models]
x = np.arange(n_models)

# 1. Accuracy & F1
w = 0.25
axes[0].bar(x - w, comp_df['Test Accuracy'], w, label='Accuracy',   color='#1565C0', alpha=0.85)
axes[0].bar(x,     comp_df['F1 Macro'],      w, label='F1 Macro',   color='#2E7D32', alpha=0.85)
axes[0].bar(x + w, comp_df['F1 Weighted'],   w, label='F1 Weighted',color='#E65100', alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(models, rotation=15, fontsize=9)
axes[0].set_ylabel('Score (%)')
axes[0].set_ylim([max(0, comp_df['F1 Macro'].min()-5), 102])
axes[0].set_title('Accuracy & F1 Scores', fontweight='bold')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3, axis='y')
for i, (acc, f1) in enumerate(zip(comp_df['Test Accuracy'], comp_df['F1 Macro'])):
    axes[0].text(i-w, acc+0.3, f'{acc:.1f}', ha='center', fontsize=7)
    axes[0].text(i,   f1 +0.3, f'{f1:.1f}',  ha='center', fontsize=7)

# 2. Cross-validation F1 with error bars
axes[1].bar(x, comp_df['CV F1 Mean'], color=bar_colors, alpha=0.85,
             yerr=comp_df['CV F1 Std'], capsize=5, edgecolor='black', lw=0.5)
axes[1].set_xticks(x)
axes[1].set_xticklabels(models, rotation=15, fontsize=9)
axes[1].set_ylabel('CV F1 Macro (%)')
axes[1].set_title('5-Fold Cross-Validation F1\n(error bars = ±1 std)', fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

# 3. Training time
axes[2].bar(x, comp_df['Train Time (s)'], color=bar_colors, alpha=0.85,
             edgecolor='black', lw=0.5)
axes[2].set_xticks(x)
axes[2].set_xticklabels(models, rotation=15, fontsize=9)
axes[2].set_ylabel('Training Time (seconds)')
axes[2].set_title('Training Time Comparison', fontweight='bold')
axes[2].grid(True, alpha=0.3, axis='y')
for i, t in enumerate(comp_df['Train Time (s)']):
    axes[2].text(i, t+0.1, f'{t:.1f}s', ha='center', fontsize=9)

plt.suptitle('Model Comparison — Temperature Fault Detection',
              fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'model_05_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print('📁 Saved: model_05_comparison.png')

---
## 7. Best Model — Deep Dive Evaluation

Full evaluation of the top-performing model:  
confusion matrix, per-class metrics, and ROC curves.

In [ ]:
# Get best model result
best_result = [r for r in results_log if r['Model'] == best_model_name][0]
best_model  = best_result['model_obj']
y_pred_best = best_result['y_pred']

# Pick correct X_test for this model
X_test_best = X_test_s if best_model_name in ['KNN (k=7)', 'SVM (RBF kernel)'] else X_test

print(f'Best model: {best_model_name}')
print(f'Test Accuracy: {best_result["Test Accuracy"]}%')
print(f'F1 Macro     : {best_result["F1 Macro"]}%')

In [ ]:
# ── Normalized confusion matrix ──
cm     = confusion_matrix(y_test, y_pred_best)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
             xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
             linewidths=0.5, ax=axes[0], cbar=False,
             annot_kws={'size':12, 'weight':'bold'})
axes[0].set_title(f'{best_model_name}\nConfusion Matrix (counts)',
                   fontsize=12, fontweight='bold')
axes[0].set_xlabel('Predicted', fontsize=10)
axes[0].set_ylabel('Actual', fontsize=10)
axes[0].tick_params(axis='x', rotation=20)

# Normalized %
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues',
             xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
             linewidths=0.5, ax=axes[1],
             cbar_kws={'label':'% of actual class'},
             annot_kws={'size':12, 'weight':'bold'})
axes[1].set_title(f'{best_model_name}\nConfusion Matrix (% per class)',
                   fontsize=12, fontweight='bold')
axes[1].set_xlabel('Predicted', fontsize=10)
axes[1].set_ylabel('Actual', fontsize=10)
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'model_06_cm_normalized.png'), dpi=150, bbox_inches='tight')
plt.show()
print('📁 Saved: model_06_cm_normalized.png')

print('\n📌 Reading the matrix:')
print('  Diagonal = correctly classified')
print('  Off-diagonal = misclassifications')
print('  Row = actual class, Column = predicted class')

In [ ]:
# ── ROC Curves — one per class (One-vs-Rest) ──
from sklearn.preprocessing import label_binarize

y_test_bin = label_binarize(y_test, classes=[0,1,2,3])

# Get probability scores
if hasattr(best_model, 'predict_proba'):
    y_prob = best_model.predict_proba(X_test_best)
elif hasattr(best_model, 'decision_function'):
    y_prob = best_model.decision_function(X_test_best)
else:
    print('Model does not support probability output — skipping ROC curves.')
    y_prob = None

if y_prob is not None:
    fig, ax = plt.subplots(figsize=(9, 7))

    for i, (cls_id, cls_name) in enumerate(CLASS_LABELS.items()):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_prob[:, i])
        auc = roc_auc_score(y_test_bin[:, i], y_prob[:, i])
        ax.plot(fpr, tpr, lw=2.5, color=list(CLASS_COLORS.values())[i],
                 label=f'{cls_name}  (AUC = {auc:.4f})')

    ax.plot([0,1],[0,1],'k--', lw=1.2, label='Random classifier (AUC=0.50)')
    ax.set_xlabel('False Positive Rate', fontsize=11)
    ax.set_ylabel('True Positive Rate', fontsize=11)
    ax.set_title(f'ROC Curves — {best_model_name}\n(One-vs-Rest per class)',
                  fontsize=12, fontweight='bold')
    ax.legend(fontsize=9, loc='lower right')
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR,'model_07_roc.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('📁 Saved: model_07_roc.png')

    # Macro AUC
    macro_auc = roc_auc_score(y_test_bin, y_prob, average='macro', multi_class='ovr')
    print(f'\n  Macro-average AUC : {macro_auc:.4f}')
    print('  (1.0 = perfect, 0.5 = random guessing)')

In [ ]:
# ── Per-class Precision / Recall / F1 bar chart ──
from sklearn.metrics import precision_recall_fscore_support

prec, rec, f1, sup = precision_recall_fscore_support(
    y_test, y_pred_best, labels=[0,1,2,3])

fig, ax = plt.subplots(figsize=(13, 5))
x  = np.arange(4)
w  = 0.25
ax.bar(x - w, prec * 100, w, label='Precision', color='#1565C0', alpha=0.85)
ax.bar(x,     rec  * 100, w, label='Recall',    color='#2E7D32', alpha=0.85)
ax.bar(x + w, f1   * 100, w, label='F1 Score',  color='#E65100', alpha=0.85)

for i in range(4):
    ax.text(i-w, prec[i]*100+0.5, f'{prec[i]*100:.1f}', ha='center', fontsize=8)
    ax.text(i,   rec[i] *100+0.5, f'{rec[i] *100:.1f}', ha='center', fontsize=8)
    ax.text(i+w, f1[i]  *100+0.5, f'{f1[i]  *100:.1f}', ha='center', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, fontsize=10)
ax.set_ylabel('Score (%)')
ax.set_ylim([0, 108])
ax.set_title(f'Per-Class Precision / Recall / F1 — {best_model_name}',
              fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'model_08_per_class.png'), dpi=150, bbox_inches='tight')
plt.show()
print('📁 Saved: model_08_per_class.png')

---
## 8. Feature Importance

Which features does the best model rely on most?  
This tells us which signal characteristics are most useful for detecting temperature faults.

In [ ]:
# Random Forest and XGBoost both provide feature_importances_
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    imp_df = pd.DataFrame({'Feature': FINAL_FEATS, 'Importance': importances})
    imp_df = imp_df.sort_values('Importance', ascending=False).reset_index(drop=True)

    print(f'📊 FEATURE IMPORTANCE — {best_model_name}')
    for _, row in imp_df.iterrows():
        bar = '█' * int(row['Importance'] * 100)
        print(f'  {row["Feature"]:18s}  {row["Importance"]:.4f}  {bar}')

    fig, ax = plt.subplots(figsize=(11, 6))
    colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(imp_df))[::-1])
    bars = ax.barh(imp_df['Feature'], imp_df['Importance'],
                    color=colors, edgecolor='black', linewidth=0.5)
    for bar_i, val in zip(bars, imp_df['Importance']):
        ax.text(val + 0.002, bar_i.get_y() + bar_i.get_height()/2,
                 f'{val:.4f}', va='center', fontsize=9)
    ax.set_title(f'Feature Importance — {best_model_name}\n(how much each feature contributes to correct predictions)',
                  fontsize=12, fontweight='bold')
    ax.set_xlabel('Importance Score')
    ax.invert_yaxis()
    ax.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR,'model_09_feat_importance.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('📁 Saved: model_09_feat_importance.png')

else:
    print(f'⚠️  {best_model_name} does not provide feature importance.')
    print('   Use Random Forest or XGBoost result for feature importance.')
    imp_df = None

In [ ]:
# ── Error Analysis — where does the model go wrong? ──
errors = y_test != y_pred_best
print(f'Total misclassifications: {errors.sum():,} / {len(y_test):,}  ({errors.mean()*100:.2f}%)')
print()
print('Misclassification breakdown (what was actual → what model predicted):')

error_pairs = {}
for actual, predicted in zip(y_test[errors], y_pred_best[errors]):
    key = f'{CLASS_LABELS[actual]} → {CLASS_LABELS[predicted]}'
    error_pairs[key] = error_pairs.get(key, 0) + 1

for pair, count in sorted(error_pairs.items(), key=lambda x: -x[1]):
    pct = count / errors.sum() * 100
    print(f'  {pair:45s}  {count:>5,}  ({pct:.1f}% of errors)')

---
## 9. Save Best Model

In [ ]:
# Save best model
model_path = os.path.join(OUTPUT_DIR, 'best_model.pkl')
with open(model_path, 'wb') as f:
    pickle.dump(best_model, f)
print(f'✅ Best model saved    → {model_path}')

# Save all models
all_models_path = os.path.join(OUTPUT_DIR, 'all_models.pkl')
all_models = {r['Model']: r['model_obj'] for r in results_log}
with open(all_models_path, 'wb') as f:
    pickle.dump(all_models, f)
print(f'✅ All models saved    → {all_models_path}')

# Save comparison table
comp_path = os.path.join(OUTPUT_DIR, 'model_comparison.csv')
comp_df[comp_cols].to_csv(comp_path, index=False)
print(f'✅ Comparison table    → {comp_path}')

# Save Phase 4 config
p4_cfg = {
    'best_model_name' : best_model_name,
    'best_accuracy'   : best_result['Test Accuracy'],
    'best_f1_macro'   : best_result['F1 Macro'],
    'FINAL_FEATS'     : FINAL_FEATS,
    'CLASS_LABELS'    : CLASS_LABELS,
    'BASE_PATH'       : BASE_PATH,
    'OUTPUT_DIR'      : OUTPUT_DIR,
    'model_path'      : model_path,
    'all_models_path' : all_models_path,
}
cfg_out = os.path.join(OUTPUT_DIR, 'temp_phase4_config.json')
with open(cfg_out, 'w') as f:
    json.dump(p4_cfg, f, indent=2)
print(f'✅ Phase 4 config      → {cfg_out}')

# ── Final Summary ──
print('\n' + '='*60)
print('PHASE 4 COMPLETE — MODEL TRAINING & EVALUATION')
print('='*60)
print(f'  Models trained : {len(results_log)}')
print()
for r in sorted(results_log, key=lambda x: -x['F1 Macro']):
    medal = '🥇' if r['Model']==best_model_name else '  '
    print(f'  {medal} {r["Model"]:25s}  Acc={r["Test Accuracy"]:6.2f}%  F1={r["F1 Macro"]:6.2f}%')
print()
print(f'  🏆 Best : {best_model_name}')
print(f'     Accuracy  : {best_result["Test Accuracy"]}%')
print(f'     F1 Macro  : {best_result["F1 Macro"]}%')
print(f'     CV F1     : {best_result["CV F1 Mean"]}% ± {best_result["CV F1 Std"]}%')
print()
print('  Project completed for Temperature sensor!')
print('  Next step: expand to Accelerometer sensor (Phase 1 → Phase 4 again)')